# Day 13 — Feature Engineering

## EduPro Predictive Modeling

### Objective
Create meaningful engineered features to improve predictive modeling,
with particular focus on EnrollmentCount while preserving the strong
CourseRevenue modeling foundation.

### Targets
- EnrollmentCount
- CourseRevenue

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# ============================================================
# DAY 13 PATH CONFIGURATION
# ============================================================

feature_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data"
)

model_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data"
)

# Day 8 source dataset
day8_file_path = (
    feature_folder /
    "EduPro_Day8_Prediction_Targets.xlsx"
)

# Day 12 evaluation reference
day12_file_path = (
    model_folder /
    "EduPro_Day12_Model_Evaluation.xlsx"
)

# Day 13 output
day13_file_path = (
    feature_folder /
    "EduPro_Day13_Feature_Engineering.xlsx"
)

feature_folder.mkdir(
    parents=True,
    exist_ok=True
)

model_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Day 8 input:")
print(day8_file_path)

print("\nDay 12 evaluation:")
print(day12_file_path)

print("\nDay 13 output:")
print(day13_file_path)

Day 8 input:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day8_Prediction_Targets.xlsx

Day 12 evaluation:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day12_Model_Evaluation.xlsx

Day 13 output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx


## Validate input files

In [3]:
print("========== INPUT FILE VALIDATION ==========")

print("\nDay 8 file exists:")
print(day8_file_path.exists())

print("\nDay 12 file exists:")
print(day12_file_path.exists())

if not day8_file_path.exists():
    raise FileNotFoundError(
        f"Day 8 input file not found:\n{day8_file_path}"
    )

if not day12_file_path.exists():
    raise FileNotFoundError(
        f"Day 12 evaluation file not found:\n{day12_file_path}"
    )    

========== INPUT FILE VALIDATION ==========

Day 8 file exists:
True

Day 12 file exists:
True


## Load source data

In [4]:
prediction_targets = pd.read_excel(
    day8_file_path,
    sheet_name="Prediction_Targets"
)

print("Day 8 Prediction_Targets loaded successfully.")

print("\nShape:")
print(prediction_targets.shape)

print("\nColumns:")
print(prediction_targets.columns.tolist())

Day 8 Prediction_Targets loaded successfully.

Shape:
(60, 15)

Columns:
['CourseID', 'CourseName', 'CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherID', 'TeacherName', 'TeacherRating', 'YearsOfExperience', 'Expertise', 'EnrollmentCount', 'CourseRevenue']


In [5]:
prediction_targets.head()

,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,TeacherID,TeacherName,TeacherRating,YearsOfExperience,Expertise,EnrollmentCount,CourseRevenue
0,CR00050,Computer Vision,Artificial Intelligence,Paid,Beginner,490.9,7.55,4.55,TC00040,Kimberly Miller,4.58,24,Cybersecurity,174,85416.6
1,CR00021,Data Analysis with Python,Data Science,Free,Intermediate,0.0,1.20,3.60,TC00016,David Carlson,2.92,1,Data Science,196,0.0
2,CR00009,Web Design Fundamentals,Design,Free,Beginner,0.0,48.19,4.51,TC00051,John Obrien,1.77,2,Design,155,0.0
3,CR00022,Data Visualization,Data Science,Free,Beginner,0.0,32.64,3.65,TC00010,Frances Sanchez,2.18,1,Data Science,177,0.0
4,CR00027,Neural Networks,Machine Learning,Free,Advanced,0.0,9.30,1.81,TC00036,Brenda Mclean,1.39,4,Digital Marketing,152,0.0


In [6]:
print("========== SOURCE DATA VALIDATION ==========")

print("\nShape:")
print(prediction_targets.shape)

print("\nMissing values:")
print(prediction_targets.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(prediction_targets["CourseID"].duplicated().sum())

print("\nUnique CourseIDs:")
print(prediction_targets["CourseID"].nunique())

========== SOURCE DATA VALIDATION ==========

Shape:
(60, 15)

Missing values:
0

Duplicate CourseIDs:
0

Unique CourseIDs:
60


## Preserve original data

In [7]:
feature_data = prediction_targets.copy()

print("Feature engineering dataset created.")

print("Shape:")
print(feature_data.shape)

Feature engineering dataset created.
Shape:
(60, 15)


## Create engineered numeric features

### Price and duration features

In [8]:
# ============================================================
# PRICE / DURATION FEATURES
# ============================================================

feature_data["PricePerDay"] = (
    feature_data["CoursePrice"] /
    feature_data["CourseDuration"].replace(0, np.nan)
)

feature_data["PricePerDay"] = (
    feature_data["PricePerDay"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

feature_data["PriceSquared"] = (
    feature_data["CoursePrice"] ** 2
)

feature_data["DurationSquared"] = (
    feature_data["CourseDuration"] ** 2
)

feature_data["LogCoursePrice"] = np.log1p(
    feature_data["CoursePrice"]
)

feature_data["LogCourseDuration"] = np.log1p(
    feature_data["CourseDuration"]
)

print("Price and duration features created.")

Price and duration features created.


## Create quality features

### Rating features

In [9]:
# ============================================================
# QUALITY / RATING FEATURES
# ============================================================

feature_data["RatingGap"] = (
    feature_data["CourseRating"] -
    feature_data["TeacherRating"]
)

feature_data["AverageRating"] = (
    feature_data["CourseRating"] +
    feature_data["TeacherRating"]
) / 2

feature_data["CourseQualityScore"] = (
    feature_data["CourseRating"] *
    feature_data["TeacherRating"]
)

feature_data["ExperienceRatingScore"] = (
    feature_data["YearsOfExperience"] *
    feature_data["TeacherRating"]
)

print("Rating and quality features created.")

Rating and quality features created.


## Create pricing-quality relationship

### Price × quality features

In [10]:
# ============================================================
# PRICE / QUALITY INTERACTION FEATURES
# ============================================================

feature_data["PriceRatingInteraction"] = (
    feature_data["CoursePrice"] *
    feature_data["AverageRating"]
)

feature_data["PricePerRatingPoint"] = (
    feature_data["CoursePrice"] /
    feature_data["AverageRating"].replace(0, np.nan)
)

feature_data["PricePerRatingPoint"] = (
    feature_data["PricePerRatingPoint"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Price-quality interaction features created.")

Price-quality interaction features created.


## Create categorical interaction features

### Category/type/level interactions

In [11]:
# ============================================================
# CATEGORICAL INTERACTION FEATURES
# ============================================================

feature_data["Category_Type"] = (
    feature_data["CourseCategory"].astype(str)
    + "_" +
    feature_data["CourseType"].astype(str)
)

feature_data["Category_Level"] = (
    feature_data["CourseCategory"].astype(str)
    + "_" +
    feature_data["CourseLevel"].astype(str)
)

feature_data["Type_Level"] = (
    feature_data["CourseType"].astype(str)
    + "_" +
    feature_data["CourseLevel"].astype(str)
)

feature_data["Category_Expertise"] = (
    feature_data["CourseCategory"].astype(str)
    + "_" +
    feature_data["Expertise"].astype(str)
)

feature_data["Level_Expertise"] = (
    feature_data["CourseLevel"].astype(str)
    + "_" +
    feature_data["Expertise"].astype(str)
)

print("Categorical interaction features created.")

Categorical interaction features created.


## Create experience bands

### Teacher experience bands

In [12]:
# ============================================================
# EXPERIENCE BAND
# ============================================================

feature_data["ExperienceBand"] = pd.cut(
    feature_data["YearsOfExperience"],
    bins=[-np.inf, 5, 10, 20, np.inf],
    labels=[
        "Early",
        "Developing",
        "Experienced",
        "Highly_Experienced"
    ]
)

print(
    feature_data[
        [
            "YearsOfExperience",
            "ExperienceBand"
        ]
    ].head(10)
)

   YearsOfExperience      ExperienceBand
0                 24  Highly_Experienced
1                  1               Early
2                  2               Early
3                  1               Early
4                  4               Early
5                 24  Highly_Experienced
6                  9          Developing
7                 24  Highly_Experienced
8                  4               Early
9                 21  Highly_Experienced


## Insoect engineered features

### Features list

In [13]:
original_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

engineered_features = [
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

print("Original modeling features:")
print(original_features)

print("\nEngineered features:")
print(engineered_features)

print("\nNumber of original features:")
print(len(original_features))

print("\nNumber of engineered features:")
print(len(engineered_features))

Original modeling features:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'Expertise']

Engineered features:
['PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint', 'Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

Number of original features:
9

Number of engineered features:
17


## Check feature quality

### Missing and infinite values

In [14]:
original_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

engineered_features = [
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

print("Original modeling features:")
print(original_features)

print("\nEngineered features:")
print(engineered_features)

print("\nNumber of original features:")
print(len(original_features))

print("\nNumber of engineered features:")
print(len(engineered_features))

Original modeling features:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'Expertise']

Engineered features:
['PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint', 'Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

Number of original features:
9

Number of engineered features:
17


### Engineered feature preview

In [15]:
feature_data[
    [
        "CourseID",
        "CoursePrice",
        "CourseDuration",
        "CourseRating",
        "TeacherRating",
        "YearsOfExperience",
        "PricePerDay",
        "RatingGap",
        "AverageRating",
        "CourseQualityScore",
        "ExperienceRatingScore",
        "PriceRatingInteraction"
    ]
].head(10)

,CourseID,CoursePrice,CourseDuration,CourseRating,TeacherRating,YearsOfExperience,PricePerDay,RatingGap,AverageRating,CourseQualityScore,ExperienceRatingScore,PriceRatingInteraction
0,CR00050,490.9,7.55,4.55,4.58,24,65.019868,-0.03,4.565,20.8390,109.92,2240.9585
1,CR00021,0.0,1.20,3.60,2.92,1,0.000000,0.68,3.260,10.5120,2.92,0.0000
2,CR00009,0.0,48.19,4.51,1.77,2,0.000000,2.74,3.140,7.9827,3.54,0.0000
3,CR00022,0.0,32.64,3.65,2.18,1,0.000000,1.47,2.915,7.9570,2.18,0.0000
4,CR00027,0.0,9.30,1.81,1.39,4,0.000000,0.42,1.600,2.5159,5.56,0.0000
5,CR00036,0.0,15.75,4.68,4.58,24,0.000000,0.10,4.630,21.4344,109.92,0.0000
6,CR00018,0.0,15.44,2.01,4.29,9,0.000000,-2.28,3.150,8.6229,38.61,0.0000
7,CR00037,0.0,33.93,3.45,4.58,24,0.000000,-1.13,4.015,15.8010,109.92,0.0000
8,CR00060,0.0,8.95,2.14,1.39,4,0.000000,0.75,1.765,2.9746,5.56,0.0000
9,CR00053,0.0,40.07,2.67,4.97,21,0.000000,-2.30,3.820,13.2699,104.37,0.0000


## Check correlations

### Numeric feature correlations

In [16]:
numeric_columns = feature_data.select_dtypes(
    include=np.number
).columns

correlation_matrix = feature_data[
    numeric_columns
].corr()

print("Correlation with EnrollmentCount:")
print(
    correlation_matrix["EnrollmentCount"]
    .sort_values(ascending=False)
)

Correlation with EnrollmentCount:
EnrollmentCount           1.000000
CourseRating              0.294010
RatingGap                 0.243011
CourseQualityScore        0.196366
AverageRating             0.176756
TeacherRating            -0.034101
PricePerDay              -0.077715
DurationSquared          -0.077833
ExperienceRatingScore    -0.080487
PriceSquared             -0.089899
YearsOfExperience        -0.096810
CourseDuration           -0.103414
CourseRevenue            -0.116611
PriceRatingInteraction   -0.130611
CoursePrice              -0.163356
PricePerRatingPoint      -0.177110
LogCourseDuration        -0.178701
LogCoursePrice           -0.215061
Name: EnrollmentCount, dtype: float64


### Revenue correlations

In [17]:
print("Correlation with CourseRevenue:")

print(
    correlation_matrix["CourseRevenue"]
    .sort_values(ascending=False)
)

Correlation with CourseRevenue:
CourseRevenue             1.000000
CoursePrice               0.996722
PriceSquared              0.968414
PriceRatingInteraction    0.960085
PricePerRatingPoint       0.930704
LogCoursePrice            0.899657
PricePerDay               0.653643
YearsOfExperience         0.071037
ExperienceRatingScore     0.065923
TeacherRating             0.039248
CourseQualityScore        0.021804
AverageRating             0.014147
CourseRating             -0.018298
RatingGap                -0.042534
LogCourseDuration        -0.068669
DurationSquared          -0.083077
CourseDuration           -0.094198
EnrollmentCount          -0.116611
Name: CourseRevenue, dtype: float64


## Compare original vs engineered feature counts

In [18]:
all_modeling_features = (
    original_features +
    engineered_features
)

print("Original feature count:")
print(len(original_features))

print("\nEngineered feature count:")
print(len(engineered_features))

print("\nTotal modeling feature count:")
print(len(all_modeling_features))

Original feature count:
9

Engineered feature count:
17

Total modeling feature count:
26


## Create features dictionary 

In [19]:
feature_dictionary = pd.DataFrame({
    "Feature": engineered_features,
    "Feature_Type": [
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Categorical Interaction",
        "Categorical Interaction",
        "Categorical Interaction",
        "Categorical Interaction",
        "Categorical Interaction",
        "Categorical"
    ],
    "Purpose": [
        "Course price relative to duration",
        "Capture nonlinear price effects",
        "Capture nonlinear duration effects",
        "Reduce price scale/skew",
        "Reduce duration scale/skew",
        "Difference between course and teacher rating",
        "Combined course and teacher rating",
        "Combined quality interaction",
        "Teacher experience combined with rating",
        "Price relative to combined quality",
        "Price relative to rating",
        "Course category and course type interaction",
        "Course category and course level interaction",
        "Course type and course level interaction",
        "Course category and teacher expertise interaction",
        "Course level and teacher expertise interaction",
        "Teacher experience group"
    ]
})

feature_dictionary

,Feature,Feature_Type,Purpose
0,PricePerDay,Numeric,Course price relative to duration
1,PriceSquared,Numeric,Capture nonlinear price effects
2,DurationSquared,Numeric,Capture nonlinear duration effects
3,LogCoursePrice,Numeric,Reduce price scale/skew
4,LogCourseDuration,Numeric,Reduce duration scale/skew
5,RatingGap,Numeric,Difference between course and teacher rating
6,AverageRating,Numeric,Combined course and teacher rating
7,CourseQualityScore,Numeric,Combined quality interaction
8,ExperienceRatingScore,Numeric,Teacher experience combined with rating
9,PriceRatingInteraction,Numeric,Price relative to combined quality


## Final Validation

In [20]:
print("==============================================")
print("DAY 13 FEATURE ENGINEERING VALIDATION")
print("==============================================")

print("\nOriginal dataset shape:")
print(prediction_targets.shape)

print("\nEngineered dataset shape:")
print(feature_data.shape)

print("\nOriginal columns:")
print(len(prediction_targets.columns))

print("\nEngineered columns:")
print(len(feature_data.columns))

print("\nTarget columns:")
print(target_columns)

print("\nNumber of engineered features:")
print(len(engineered_features))

print("\nMissing values in engineered dataset:")
print(feature_data.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(feature_data["CourseID"].duplicated().sum())

DAY 13 FEATURE ENGINEERING VALIDATION

Original dataset shape:
(60, 15)

Engineered dataset shape:
(60, 32)

Original columns:
15

Engineered columns:
32

Target columns:
['EnrollmentCount', 'CourseRevenue']

Number of engineered features:
17

Missing values in engineered dataset:
0

Duplicate CourseIDs:
0


## Save Day 13

In [21]:
# ============================================================
# SAVE DAY 13 FEATURE ENGINEERING OUTPUT
# ============================================================

with pd.ExcelWriter(
    day13_file_path,
    engine="openpyxl"
) as writer:

    feature_data.to_excel(
        writer,
        sheet_name="Engineered_Features",
        index=False
    )

    feature_dictionary.to_excel(
        writer,
        sheet_name="Feature_Dictionary",
        index=False
    )

    correlation_matrix.to_excel(
        writer,
        sheet_name="Feature_Correlations"
    )

print("Day 13 feature engineering results saved successfully:")
print(day13_file_path)

Day 13 feature engineering results saved successfully:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx


## Reload and validate saved workbook

In [22]:
# ============================================================
# RELOAD DAY 13 OUTPUT
# ============================================================

check_features = pd.read_excel(
    day13_file_path,
    sheet_name="Engineered_Features"
)

check_dictionary = pd.read_excel(
    day13_file_path,
    sheet_name="Feature_Dictionary"
)

print("========== DAY 13 OUTPUT VALIDATION ==========")

print("\nEngineered Features shape:")
print(check_features.shape)

print("\nFeature Dictionary shape:")
print(check_dictionary.shape)

print("\nOutput file exists:")
print(day13_file_path.exists())

========== DAY 13 OUTPUT VALIDATION ==========

Engineered Features shape:
(60, 32)

Feature Dictionary shape:
(17, 3)

Output file exists:
True


## Final completion cell

In [23]:
print("==============================================")
print("DAY 13 FEATURE ENGINEERING COMPLETED")
print("==============================================")

print("\nInput:")
print(day8_file_path)

print("\nDay 12 evaluation reference:")
print(day12_file_path)

print("\nOutput:")
print(day13_file_path)

print("\nOriginal rows:")
print(len(prediction_targets))

print("\nEngineered rows:")
print(len(feature_data))

print("\nOriginal columns:")
print(len(prediction_targets.columns))

print("\nEngineered columns:")
print(len(feature_data.columns))

print("\nEngineered features created:")
print(len(engineered_features))

print("\nMissing values:")
print(feature_data.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(feature_data["CourseID"].duplicated().sum())

print("\nOutput file exists:")
print(day13_file_path.exists())

print("\n==============================================")
print("DAY 13 COMPLETED")
print("==============================================")

DAY 13 FEATURE ENGINEERING COMPLETED

Input:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day8_Prediction_Targets.xlsx

Day 12 evaluation reference:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day12_Model_Evaluation.xlsx

Output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx

Original rows:
60

Engineered rows:
60

Original columns:
15

Engineered columns:
32

Engineered features created:
17

Missing values:
0

Duplicate CourseIDs:
0

Output file exists:
True

DAY 13 COMPLETED
